# 📈 Notebook 02 — EDA: Sentiment vs PnL

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 12, 'figure.dpi': 150})
sns.set_style('whitegrid')

In [ ]:
closed = pd.read_csv('../data/closed_trades.csv')
closed_clean = pd.read_csv('../data/closed_trades_clean.csv')

S_ORDER = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
S_ORDER = [s for s in S_ORDER if s in closed['sentiment_class'].unique()]
# Red = Fear, Orange = Neutral, Green = Greed
S_PAL = {'Extreme Fear':'#d32f2f', 'Fear':'#f44336', 'Neutral':'#ff9800', 'Greed':'#4caf50', 'Extreme Greed':'#388e3c'}

## 5 EDA — Sentiment vs PnL (Violin, Boxplot, Win Rate)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.violinplot(data=closed_clean, x='sentiment_class', y='Closed PnL', order=S_ORDER, palette=S_PAL, ax=axes[0])
axes[0].set_title('PnL Distribution per Sentiment (Violin Plot)')

wr = closed.groupby('sentiment_class')['Is Profitable'].mean().reindex(S_ORDER) * 100
axes[1].barh(S_ORDER[::-1], wr[::-1].values, color=[S_PAL[s] for s in S_ORDER[::-1]], edgecolor='white')
axes[1].set_title('Win Rate per Sentiment (%)')
axes[1].set_xlabel('Win Rate (%)')
for i, v in enumerate(wr[::-1].values): axes[1].text(v + 0.5, i, f'{v:.1f}%', va='center')
plt.tight_layout()
plt.savefig('../outputs/pnl_vs_sentiment_eda.png')
plt.show()

## 6 EDA — Direction Analysis (Long vs Short)

In [ ]:
ls = closed[closed['Trade Type'].isin(['Long', 'Short'])].copy()
ls_ct = ls.groupby(['sentiment_class', 'Trade Type']).size().unstack().reindex(S_ORDER).fillna(0)
ls_pct = ls_ct.div(ls_ct.sum(axis=1), axis=0) * 100
ls_pct.plot(kind='bar', stacked=True, color=['#42a5f5', '#ef5350'], figsize=(10,5))
plt.title('Long vs Short Ratio per Sentiment')
plt.ylabel('%'); plt.xticks(rotation=15)
plt.savefig('../outputs/long_short_ratio.png')
plt.show()

## 7 Account-Level Analysis

In [ ]:
acct_sent = closed.groupby(['Account', 'sentiment_class']).agg(
    avg_pnl=('Closed PnL', 'mean'), win_rate=('Is Profitable', 'mean'), trades=('Closed PnL', 'count')
)
print(acct_sent.head(10))

## Heatmap: Account x Sentiment

In [ ]:
pnl_heat = closed_clean.groupby(['Account', 'sentiment_class'])['Closed PnL'].mean().unstack().reindex(columns=S_ORDER).fillna(0)
pnl_heat.index = [f'T{i}' for i in range(len(pnl_heat))]
plt.figure(figsize=(10,12))
sns.heatmap(pnl_heat, cmap='RdYlGn', center=0, cbar_kws={'label': 'Avg PnL'})
plt.title('Account x Sentiment -> Avg PnL')
plt.savefig('../outputs/heatmap_account_sentiment.png')
plt.show()

## 8 Correlation Analysis

In [ ]:
daily = closed.groupby('date').agg(avg_pnl=('Closed PnL', 'mean'), sentiment=('sentiment_value', 'first')).dropna()
sp_corr, p_val = stats.spearmanr(daily['sentiment'], daily['avg_pnl'])

plt.figure(figsize=(8,6))
sns.regplot(data=daily, x='sentiment', y='avg_pnl', scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
plt.title(f'Daily Sentiment vs Avg PnL (Spearman: {sp_corr:.3f}, p: {p_val:.3f})')
plt.savefig('../outputs/correlation_scatter.png')
plt.show()

## 9 Statistical Testing (Fear vs Greed)

In [ ]:
fear_pnl = closed_clean[closed_clean['sentiment_class'].str.contains('Fear', na=False)]['Closed PnL']
greed_pnl = closed_clean[closed_clean['sentiment_class'].str.contains('Greed', na=False)]['Closed PnL']
t_stat, p_val = stats.ttest_ind(fear_pnl, greed_pnl, equal_var=False)
print(f'T-test Fear vs Greed PnL: t={t_stat:.4f}, p={p_val:.6f}')

## Fee Erosion Analysis

In [ ]:
total_fees = closed['Fee'].sum()
print(f'Total Fees paid on closed trades: ${total_fees:,.2f}')
print(f'Avg fee per trade: ${closed["Fee"].mean():.2f}')